In [1]:
import pandas
import duckdb
from IPython.core.magic import register_cell_magic
from IPython import get_ipython

In [2]:
conn = duckdb.connect("ehr.duckdb")

In [3]:
@register_cell_magic
def duck(line, cell):
    name = line.strip() or "_result"
    result = conn.execute(cell).df()
    get_ipython().push({name: result})
    return result.style

In [4]:
# Load demo_queries.sql and split it with DuckDB's own parser, so the SQL has
# exactly one home. Re-run this cell after editing the .sql file.
import re
from pathlib import Path

QUERIES = {}
for _st in duckdb.extract_statements(Path("demo_queries.sql").read_text()):
    _sql = _st.query.strip()
    _m = re.search(r"^-- (\d+)\. (.+?) -*$", _sql, re.M)
    if _m:
        QUERIES[int(_m.group(1))] = (_m.group(2).strip(), _sql)

def q(n, name=None):
    """Run demo query n. Pass name= to also bind the DataFrame to that variable."""
    title, sql = QUERIES[n]
    print(f"{n}. {title}")
    df = conn.execute(sql).df()
    if name:
        get_ipython().push({name: df})
    return df.style

print(f"loaded {len(QUERIES)} queries from demo_queries.sql\n")
for _n, (_t, _) in sorted(QUERIES.items()):
    print(f"  q({_n:>2})  {_t}")

loaded 13 queries from demo_queries.sql

  q( 1)  Panel snapshot
  q( 2)  Chronic condition registry
  q( 3)  Uncontrolled hypertension
  q( 4)  Heart failure without guideline-directed therapy
  q( 5)  Atrial fibrillation without anticoagulation
  q( 6)  Lab trajectory: biggest movers
  q( 7)  Outstanding lab orders
  q( 8)  Abnormal lab burden
  q( 9)  Polypharmacy
  q(10)  Provider scorecard
  q(11)  Note search
  q(12)  Note extraction audit
  q(13)  Note template distribution


In [5]:
q(1)

1. Panel snapshot


,patients,encounters,female_enc,avg_age,medicare_age_enc,providers,date_range
0,100,153,79,44.600000,26,20,2025-05-28 .. 2026-05-24


In [6]:
q(2)

2. Chronic condition registry


,dx_name,icd10,patients,pct_of_panel
0,"Hypertensive Heart Disease, Unspecified",I11.9,25,25.000000
1,Essential Hypertension,I10,21,21.000000
2,Pure Hypercholesterolaemia,E78.00,19,19.000000
3,Pure Hypertriglyceridaemia,E78.1,18,18.000000
4,GERD without Oesophagitis,K21.9,17,17.000000
5,"Hyperlipidaemia, Unspecified",E78.5,13,13.000000
6,Type 2 DM with Diabetic Peripheral Neuropathy,E11.51,12,12.000000
7,GERD with Oesophagitis,K21.0,10,10.000000
8,"Anxiety Disorder, Unspecified",F41.9,10,10.000000
9,Subclinical Iodine-Deficiency Hypothyroidism,E02,9,9.000000


In [7]:
q(3)

3. Uncontrolled hypertension


,PAT_NAME,PAT_AGE,bp,bmi,CONTACT_DATE,visit_prov_name
0,"Cain, Jacob",46,184/95,44.600000,2025-09-22 00:00:00,Wendy Cortez
1,"Cain, Jacob",46,184/95,44.600000,2026-05-22 00:00:00,Dean Washington Jr.
2,"Bonilla, Andrew",72,181/96,18.400000,2025-10-22 00:00:00,Breanna Schmitt
3,"Ware, Cassandra",25,180/61,55.100000,2025-09-26 00:00:00,Wendy Cortez
4,"Ware, Cassandra",25,180/61,55.100000,2025-08-01 00:00:00,Gina Powell
5,"Moore, Ryan",28,180/64,23.700000,2026-05-19 00:00:00,Barbara Newman
6,"Moore, Ryan",28,180/64,23.700000,2026-04-23 00:00:00,Breanna Schmitt
7,"Lang, Amy",18,179/68,37.400000,2026-04-13 00:00:00,Tina Wells
8,"Lang, Amy",18,179/68,37.400000,2026-05-08 00:00:00,Miss Adriana Flores
9,"Young, Cynthia",41,175/63,27.100000,2026-05-14 00:00:00,Linda Short


In [8]:
q(4)

4. Heart failure without guideline-directed therapy


,PAT_NAME,PAT_AGE,med_classes,on_betablocker,on_raas_agent
0,"Sandoval, John",75,['ACEi' 'CCB' 'GIP/GLP-1 RA'],False,True
1,"Stein, Larry",43,['Fibrate' 'PPI' 'Statin'],False,False
2,"Zavala, Manuel",70,['Biguanide' 'Omega-3' 'PPI' 'SGLT2i'],False,False


In [9]:
q(5)

5. Atrial fibrillation without anticoagulation


,PAT_NAME,PAT_AGE,afib_dx
0,"Fowler, Jessica",61,Longstanding Persistent Atrial Fibrillation
1,"Payne, Ryan",53,Longstanding Persistent Atrial Fibrillation
2,"Chandler, Cassandra",52,Chronic Atrial Fibrillation
3,"Mcdaniel, Dana",46,Other Persistent Atrial Fibrillation


In [10]:
q(6)

6. Lab trajectory: biggest movers


,PAT_NAME,analyte,first_value,latest_value,delta,span
0,"Ramirez, William",NT-proBNP,45.382000,199.359000,153.980000,2025-07-21 .. 2026-01-24
1,"Mckinney, Scott",LDL Cholesterol,123.059000,8.560000,-114.500000,2024-12-01 .. 2025-06-09
2,"Stein, Larry",NT-proBNP,196.274000,85.059000,-111.220000,2025-03-08 .. 2025-09-14
3,"Cuevas, Meredith",LDL Cholesterol,14.090000,124.955000,110.870000,2025-03-15 .. 2025-11-20
4,"Ford, Gabriel",LDL Cholesterol,112.563000,14.318000,-98.250000,2024-12-26 .. 2025-10-13
5,"Young, Cynthia",LDL Cholesterol,18.631000,109.698000,91.070000,2025-02-27 .. 2025-03-31
6,"Boone, Terri",eGFR,164.614000,78.747000,-85.870000,2025-01-15 .. 2025-06-03
7,"Young, Cynthia",eGFR,162.037000,79.546000,-82.490000,2025-01-04 .. 2025-01-21
8,"Vargas, Kimberly",LDL Cholesterol,14.109000,92.046000,77.940000,2024-06-12 .. 2025-02-11
9,"Boone, Terri",LDL Cholesterol,36.623000,108.045000,71.420000,2025-01-23 .. 2025-10-03


In [11]:
q(7)

7. Outstanding lab orders


,provider,specialty,pending,tests,oldest_order
0,Tina Wells,Internal Medicine,13,"ALT, Fasting Glucose, Potassium, HDL Cholesterol, Vitamin B12, Total Cholesterol, TSH, CBC — Hemoglobin, AST",2025-06-10 00:00:00
1,David White,Gastroenterology,10,"LDL Cholesterol, Fasting Glucose, CBC — WBC, Basic Metabolic Panel — Creatinine, Magnesium, TSH, Triglycerides, Potassium",2025-06-19 00:00:00
2,Daniel Mills,Cardiology,10,"ALT, Triglycerides, LDL Cholesterol, Fasting Glucose, Vitamin B12, HbA1c, BUN, INR / PT",2025-06-06 00:00:00
3,Breanna Schmitt,Pulmonology,9,"Potassium, eGFR, Basic Metabolic Panel — Creatinine, HDL Cholesterol, LDL Cholesterol, Sodium, IgE Total, TSH",2025-06-13 00:00:00
4,Miss Adriana Flores,Neurology,8,"Eosinophil Count, ALT, Triglycerides, eGFR, Basic Metabolic Panel — Creatinine, Vitamin B12, H. pylori Ag Stool",2025-08-30 00:00:00
5,William Mccullough,Gastroenterology,8,"Eosinophil Count, ALT, BNP, CBC — Hemoglobin, Vitamin B12, Basic Metabolic Panel — Creatinine, CBC — WBC, Total Cholesterol",2025-05-27 00:00:00
6,Peter Thomas DVM,Internal Medicine,8,"Vitamin B12, CBC — Hemoglobin, H. pylori Ag Stool, Potassium, Vitamin D 25-OH, HDL Cholesterol, eGFR",2025-07-14 00:00:00
7,Tyler Hill,Psychiatry,7,"LDL Cholesterol, Fasting Glucose, Sodium, Urine Albumin/Creat Ratio, BUN, AST",2025-07-31 00:00:00
8,Michael Lee,Endocrinology,7,"LDL Cholesterol, Urine Albumin/Creat Ratio, Potassium, Fasting Glucose, Magnesium, HDL Cholesterol",2025-06-12 00:00:00
9,Barbara Newman,Family Medicine,6,"Digoxin Level, Fasting Glucose, ALT, SpO2, Urine Albumin/Creat Ratio, CBC — Hemoglobin",2025-08-29 00:00:00


In [12]:
q(8)

8. Abnormal lab burden


,analyte,results,abnormal,pct_abnormal,avg_abnormal_value,ref_range,unit
0,Eosinophil Count,13,6,46.200000,0.650000,0.05-0.5,K/uL
1,Free T4,13,5,38.500000,2.280000,0.8-1.8,ng/dL
2,Urine Albumin/Creat Ratio,59,22,37.300000,36.880000,0.0-30.0,mg/g
3,ALT,59,21,35.600000,67.220000,7.0-56.0,U/L
4,Magnesium,29,10,34.500000,2.950000,1.7-2.4,mg/dL
5,Sodium,56,19,33.900000,175.140000,136.0-145.0,mEq/L
6,LDL Cholesterol,72,24,33.300000,119.270000,0.0-100.0,mg/dL
7,Vitamin D 25-OH,45,15,33.300000,63.900000,20.0-50.0,ng/mL
8,Fasting Glucose,57,18,31.600000,124.420000,70.0-100.0,mg/dL
9,TSH,49,15,30.600000,5.010000,0.4-4.0,mIU/L


In [13]:
q(9)

9. Polypharmacy


,PAT_NAME,PAT_AGE,med_count,doses_per_day,regimen
0,"Cervantes, Stephen",19,5,4.300000,Dapagliflozin 10 mg | Metformin 1000 mg | Semaglutide 1 mg (Ozempic) | Sitagliptin 100 mg | Tirzepatide 5 mg (Mounjaro)
1,"Schmidt, Raven",66,5,5.000000,Estradiol/Levonorgestrel (Lo Loestrin FE) | Famotidine 20 mg | Medroxyprogesterone 150 mg IM | Norethindrone 0.35 mg (POP) | Omega-3 Fatty Acids 4 g
2,"Romero, Erik",49,5,5.100000,Empagliflozin 10 mg | Liraglutide 1.2 mg | Metformin 1000 mg | Sitagliptin 100 mg | Tirzepatide 5 mg (Mounjaro)
3,"Anderson, Ann",60,5,10.100000,Dupilumab 300 mg (Dupixent) | Fluticasone/Salmeterol 250/50 mcg | Losartan 50 mg | Metoprolol Succinate 50 mg | Tiotropium 18 mcg (Spiriva)
4,"Gomez, Christopher",55,5,12.000000,Atorvastatin 40 mg | Bupropion XL 150 mg | Buspirone 10 mg | Rosuvastatin 20 mg | Venlafaxine XR 75 mg
5,"Vargas, Kimberly",18,5,8.100000,Empagliflozin 10 mg | Evolocumab 140 mg (Repatha) | Omega-3 Fatty Acids 4 g | Rosuvastatin 20 mg | Sitagliptin 100 mg
6,"Miller, Nancy",24,5,12.000000,Escitalopram 10 mg | Insulin Glargine U-100 20U | Levothyroxine 50 mcg | Liothyronine 5 mcg | Liraglutide 1.2 mg
7,"Forbes, Tamara",62,5,7.000000,Apixaban 5 mg (Eliquis) | Dabigatran 150 mg (Pradaxa) | Metoprolol Succinate 50 mg | Rivaroxaban 20 mg (Xarelto) | Warfarin 5 mg
8,"Davis, Carol",75,5,10.000000,Desogestrel/Ethinyl Estradiol | Esomeprazole 40 mg | Norethindrone 0.35 mg (POP) | Pantoprazole 40 mg | Sitagliptin 100 mg
9,"Ramsey, Latoya",36,5,8.100000,Evolocumab 140 mg (Repatha) | Insulin Glargine U-100 20U | Rosuvastatin 20 mg | Sitagliptin 100 mg | Spironolactone 25 mg


In [14]:
q(10)

10. Provider scorecard


,provider,specialty,visits,awv_labs_ordered,still_pending,pct_abnormal_hx_labs
0,Peter Thomas DVM,Internal Medicine,17,67,8,25.100000
1,Daniel Mills,Cardiology,12,42,10,26.400000
2,Michael Lee,Endocrinology,10,41,7,29.900000
3,David White,Gastroenterology,10,40,10,18.400000
4,Tina Wells,Internal Medicine,9,41,13,31.800000
5,Miss Adriana Flores,Neurology,9,31,8,23.000000
6,Dean Washington Jr.,Pulmonology,8,30,3,28.100000
7,Breanna Schmitt,Pulmonology,8,30,9,28.100000
8,Tyler Hill,Psychiatry,8,35,7,28.000000
9,Barbara Newman,Family Medicine,8,29,6,35.000000


In [15]:
q(11)

11. Note search


,PAT_NAME,noted,author,bp_in_note,excerpt
0,"Erickson, Savannah",2026-05-09 00:00:00,Daniel Mills,150/99,"Annual Wellness Visit. Active conditions: Hypertensive Heart Disease, Unspecified | Genera..."
1,"Anderson, Ann",2026-02-25 00:00:00,Thomas Warren,,"Annual preventive visit. Chronic conditions: Hypertensive Heart Disease, Unspecified | Typ..."
2,"Cunningham, David",2026-02-06 00:00:00,Jordan Contreras,,"Progress Note — AWV. Problem list active: Hyperlipidaemia, Unspecified | Mild Intermittent..."
3,"Marshall, Brian",2026-02-03 00:00:00,Miss Adriana Flores,,Annual preventive visit. Chronic conditions: Essential Hypertension | Mild Intermittent As...
4,"Jackson, Shawn",2026-01-04 00:00:00,Dana Williams,,Annual preventive visit. Chronic conditions: Essential Hypertension | Major Depressive Dis...


In [16]:
q(12)

12. Note extraction audit


,field,notes,matched,pct_agreement
0,BMI,153,153,100.000000
1,blood pressure,153,153,100.000000
2,conditions,153,153,100.000000
3,medications,153,153,100.000000


In [17]:
q(13)

13. Note template distribution


,note_style,notes,patients,pct_of_notes,avg_age,avg_systolic,avg_bmi,avg_conditions,avg_meds,avg_chars
0,Annual Wellness Visit,55,50,35.900000,41.500000,148.100000,31.500000,2.350000,3.360000,352.000000
1,Annual preventive visit,41,38,26.800000,47.800000,147.400000,29.400000,2.320000,3.410000,344.000000
2,AWV encounter,33,31,21.600000,46.600000,149.200000,29.900000,2.450000,3.450000,345.000000
3,Progress Note — AWV,24,23,15.700000,43.700000,145.100000,29.500000,2.080000,3.460000,328.000000


In [18]:
# Remember to close the connection when done
conn.close()